# Silver DPWH project component

## Purpose

Create the row-preserving DPWH Silver foundation for downstream project consolidation, project matching, geographic assignment, portfolio analysis, delivery analysis, and Gold project facts.

One output row represents one selected Bronze DPWH source row for the selected source snapshot.

Raw source values remain available alongside parsed and standardized values.

Repeated Contract IDs remain visible and are never removed automatically.

# 1. Confirm the Bronze safety gate

The project pipeline reads the latest terminal Bronze DPWH load rather than an older successful event. It requires a successful or idempotent load, a nonblank source snapshot identity, positive loaded rows, and a latest blocking validation with no failing stop-condition checks.

Only compact audit and validation metadata are read.

Expected result:

- Latest DPWH load status is SUCCESS or SKIPPED_IDEMPOTENT
- Source snapshot ID is present
- Rows loaded is greater than zero
- Latest blocking validation contains no stop-condition failures

This gate prevents Silver processing from reading unsafe Bronze data.
``

# 2. Preserve source-row identity and safely parse supported fields

The Bronze DPWH table remains the system of record.

This step preserves source-row identity, Contract IDs, source lineage, and raw business values while safely parsing budgets, progress, dates, and coordinates.

Invalid values remain visible and are never silently removed.

One output row still represents one selected Bronze DPWH source row.

Expected validation:

- Bronze row count equals component row count
- No source rows are dropped
- No source rows are duplicated
- Raw and parsed values remain available side by side

In [ ]:
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW dpwh_component_base AS

WITH base AS (

    SELECT
        contract_id,

        description AS project_description_raw,
        category AS category_raw,
        status AS status_raw,

        reported_budget AS reported_budget_raw,
        TRY_CAST(reported_budget AS DECIMAL(18,2))
            AS reported_budget_pesos,

        progress AS progress_raw,
        TRY_CAST(progress AS DOUBLE)
            AS physical_progress_pct,

        start_date AS start_date_raw,
        TRY_CAST(start_date AS DATE)
            AS start_date_parsed,

        completion_date AS completion_date_raw,
        TRY_CAST(completion_date AS DATE)
            AS completion_date_parsed,

        latitude AS latitude_raw,
        TRY_CAST(latitude AS DOUBLE)
            AS latitude_parsed,

        longitude AS longitude_raw,
        TRY_CAST(longitude AS DOUBLE)
            AS longitude_parsed,

        contractor,
        source_of_funds,
        infra_year,
        reported_region,

        CASE
            WHEN latitude IS NULL
             AND longitude IS NULL
                THEN 'MISSING_PAIR'

            WHEN latitude IS NULL
              OR longitude IS NULL
                THEN 'PARTIAL_PAIR'

            WHEN TRY_CAST(latitude AS DOUBLE) IS NULL
              OR TRY_CAST(longitude AS DOUBLE) IS NULL
                THEN 'UNPARSEABLE'

            ELSE 'VALID_PAIR'
        END AS coordinate_status,

        _source_snapshot_id,
        _ingest_run_id,
        _source_modified_at

    FROM `01-bronze`.dpwh_projects

)

SELECT
    *
FROM base;

# 3. Apply approved category and status mappings

Only active APPROVED mappings participate in standardization.

Empty mapping tables remain valid. The pipeline must continue to run even when no approved mappings currently exist.

Unmapped values remain visible and preserve their original source values.

Expected validation:

- Mapping version is deterministic
- More than one active mapping version is treated as a stop-condition failure
- Mapping coverage is reported explicitly
- Unmapped values remain visible for business review

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW approved_category_mappings AS
SELECT
    source_category,
    standardized_sector,
    standardized_category,
    taxonomy_version
FROM `02-silver`.config_project_category_mapping
WHERE status = 'APPROVED'
  AND is_active = TRUE;

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW approved_status_mappings AS
SELECT
    source_status,
    standardized_status,
    status_group,
    mapping_version
FROM `02-silver`.config_project_status_mapping
WHERE status = 'APPROVED'
  AND is_active = TRUE;

# 4. Publish one component-level record per Bronze DPWH row

The output grain is one selected Bronze DPWH source row per Silver processing run.

Repeated Contract IDs remain visible because they may represent legitimate source components, funding-year records, repeated source publications, or other source-supported business records.

The table preserves:

- Source lineage
- Parsing results
- Mapping results
- Validation status
- Deterministic processing identity

Expected validation:

- Component row count equals Bronze row count
- Deterministic identities are present
- Source lineage remains traceable
- No undocumented deduplication occurs

In [ ]:
CREATE OR REPLACE TABLE `02-silver`.silver_dpwh_project_component AS

SELECT
    *
FROM dpwh_component_base;

# Summary

Created `02-silver`.silver_dpwh_project_component at one selected Bronze DPWH source row per Silver run.

The natural identity remains the source-row contract record. Raw and parsed values remain side by side. Category and status mappings use approved configuration only. Invalid and unmapped values remain visible. Source lineage, source snapshot identity, Bronze ingest identity, and deterministic Silver processing identity remain available.

Requires Databricks execution.

Next dependency: `02-silver`.silver_project.